<a href="https://colab.research.google.com/github/ailab-nda/ML/blob/main/yolo_sample_ans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO で物体検出をやってみよう（Google Colab 版）

講義「物体認識AIの発展」の実習ノートブックです。Ultralytics の **YOLO26**（2026年1月公開）を使い、学習済みモデルで次のことを体験します。

| パート | 内容 |
|---|---|
| 0 | 環境の準備 |
| 1 | 画像の物体検出（推論） |
| 2 | 検出結果の解析（枠・クラス・確信度を取り出す） |
| 3 | パラメータの影響（確信度しきい値・クラス指定・モデルサイズ） |
| 4 | 応用①：インスタンスセグメンテーション |
| 5 | 応用②：姿勢推定 |
| 6 | 応用③：動画の物体追跡 |
| 7 | 応用④：PC・スマホのカメラでリアルタイム物体認識 |
| 8 | 演習課題 |

**始める前に**：メニューの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選んでください（CPU でも動きますが、動画のパートが遅くなります）。上から順に各セルを実行（Shift + Enter）していきます。

> グラフの見出しや軸ラベルは、Colab に日本語フォントが入っていないため英語にしています。

## 0. 環境の準備

`ultralytics` は YOLO の公式ライブラリです。`lap` は物体追跡（パート6）で使う割り当てアルゴリズムのライブラリです。

In [ ]:
!pip install -q ultralytics lap

In [ ]:
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import ASSETS  # ライブラリ同梱のサンプル画像（bus.jpg, zidane.jpg）

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

ultralytics.checks()  # バージョン・GPU の有無などを表示
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("使用デバイス:", "GPU" if DEVICE == 0 else "CPU")

In [ ]:
def show(img_bgr, title=None, size=8):
    # OpenCV の画像（BGR）を matplotlib で表示する補助関数
    plt.figure(figsize=(size, size))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

BUS = str(ASSETS / "bus.jpg")
ZIDANE = str(ASSETS / "zidane.jpg")

## 1. 画像の物体検出（推論）

`YOLO("yolo26n.pt")` で COCO データセット（80クラス）で学習済みのモデルを読み込みます。初回は重みファイル（約5MB）が自動でダウンロードされます。

モデル名の `n` は nano（最小）を表します。サイズは `n` < `s` < `m` < `l` < `x` の順に大きく、精度が上がる代わりに遅くなります。

In [ ]:
model = YOLO("yolo26n.pt")

results = model(BUS, device=DEVICE, verbose=False)  # 推論。リストで返る（画像1枚なら要素1つ）
r = results[0]

show(r.plot(), "YOLO26n detection")  # plot() は枠とラベルを描き込んだ画像（BGR）を返す

## 2. 検出結果の解析

`r.boxes` に検出結果が入っています。主な属性は次のとおりです。

| 属性 | 意味 | 形 |
|---|---|---|
| `xyxy` | 枠の左上 (x1, y1) と右下 (x2, y2)（画素） | (N, 4) |
| `xywh` | 枠の中心 (x, y) と幅・高さ | (N, 4) |
| `xyxyn` | `xyxy` を画像サイズで割った正規化座標 | (N, 4) |
| `conf` | 確信度（0〜1） | (N,) |
| `cls` | クラス番号 | (N,) |

クラス番号と名前の対応は `r.names`（辞書）にあります。

In [ ]:
print("画像サイズ (高さ, 幅):", r.orig_shape)
print("検出数:", len(r.boxes))
print("クラス数:", len(r.names), "例:", {k: r.names[k] for k in range(5)})
print()
print("xyxy:\n", r.boxes.xyxy)
print("conf:", r.boxes.conf)
print("cls :", r.boxes.cls)

### 表（DataFrame）にまとめる

テンソルを NumPy 配列に変換し、pandas の表にすると扱いやすくなります（GPU 上のテンソルは `.cpu()` で CPU に移してから変換します）。

In [ ]:
def boxes_to_df(r):
    b = r.boxes
    xyxy = b.xyxy.cpu().numpy()
    df = pd.DataFrame(xyxy, columns=["x1", "y1", "x2", "y2"]).round(1)
    df.insert(0, "class", [r.names[int(c)] for c in b.cls.cpu().numpy()])
    df.insert(1, "conf", b.conf.cpu().numpy().round(3))
    df["width"] = (df.x2 - df.x1).round(1)
    df["height"] = (df.y2 - df.y1).round(1)
    df["area_ratio_%"] = (df.width * df.height / (r.orig_shape[0] * r.orig_shape[1]) * 100).round(2)
    return df

df = boxes_to_df(r)
df

In [ ]:
# クラスごとの個数
df["class"].value_counts()

### 検出結果を自分で描いてみる

`plot()` に頼らず、取り出した座標から枠を描いてみます。座標の意味を確かめるのが目的です。

In [ ]:
img = cv2.imread(BUS)
fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
for _, row in df.iterrows():
    ax.add_patch(patches.Rectangle((row.x1, row.y1), row.width, row.height,
                                   fill=False, linewidth=2, edgecolor="orange"))
    ax.text(row.x1, row.y1 - 8, f"{row['class']} {row.conf:.2f}", color="black",
            backgroundcolor="orange", fontsize=9)
ax.axis("off")
ax.set_title("Boxes drawn from r.boxes.xyxy")
plt.show()

### 2つの枠の IoU を計算する

講義で扱った IoU（Intersection over Union）を、検出された枠どうしで計算してみます。重なりがなければ 0、完全に一致すれば 1 です。

In [ ]:
def iou(a, b):
    # a, b: [x1, y1, x2, y2]
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)

boxes = df[["x1", "y1", "x2", "y2"]].values
n = len(boxes)
iou_mat = np.array([[iou(boxes[i], boxes[j]) for j in range(n)] for i in range(n)])
pd.DataFrame(iou_mat.round(2),
             index=[f"{i}:{c}" for i, c in enumerate(df["class"])],
             columns=[f"{i}:{c}" for i, c in enumerate(df["class"])])

## 3. パラメータの影響

### 3-1. 確信度しきい値 `conf`

`conf` より確信度が低い検出は捨てられます（既定値 0.25）。しきい値を下げると見逃しは減りますが誤検出が増え、上げるとその逆になります。

In [ ]:
thresholds = [0.05, 0.25, 0.5, 0.75]
fig, axes = plt.subplots(1, len(thresholds), figsize=(20, 7))
for ax, t in zip(axes, thresholds):
    rt = model(BUS, conf=t, device=DEVICE, verbose=False)[0]
    ax.imshow(cv2.cvtColor(rt.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"conf={t}: {len(rt.boxes)} boxes")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3-2. クラスを指定して検出し、切り出す

`classes=[0]` のように指定すると、そのクラス（0 = person）だけを検出します。検出した人物を画像から切り出して並べてみます。

In [ ]:
rp = model(BUS, classes=[0], device=DEVICE, verbose=False)[0]
img = cv2.imread(BUS)
crops = [img[int(y1):int(y2), int(x1):int(x2)] for x1, y1, x2, y2 in rp.boxes.xyxy.cpu().numpy()]

fig, axes = plt.subplots(1, len(crops), figsize=(3 * len(crops), 5))
for i, (ax, c) in enumerate(zip(np.atleast_1d(axes), crops)):
    ax.imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    ax.set_title(f"person #{i}  conf={float(rp.boxes.conf[i]):.2f}")
    ax.axis("off")
plt.show()

### 3-3. モデルサイズの比較

同じ画像で nano（`n`）と small（`s`）、medium（`m`）を比べます。1回目の推論は準備処理を含むため、ウォームアップしてから時間を測ります。

In [ ]:
rows = []
for name in ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt"]:
    m = YOLO(name)
    m(BUS, device=DEVICE, verbose=False)              # ウォームアップ
    rr = m(BUS, device=DEVICE, verbose=False)[0]
    params = sum(p.numel() for p in m.model.parameters()) / 1e6
    rows.append({"model": name,
                 "params (M)": round(params, 1),
                 "inference (ms)": round(rr.speed["inference"], 1),
                 "detections": len(rr.boxes),
                 "mean conf": round(float(rr.boxes.conf.mean()), 3)})
pd.DataFrame(rows)

### 3-4. 自分の画像で試す（任意）

下のセルを実行すると、ファイルを選ぶボタンが表示されます。手元の写真をアップロードして検出してみましょう。

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        ru = model(fname, device=DEVICE, verbose=False)[0]
        show(ru.plot(), fname)
        display(boxes_to_df(ru))
except ImportError:
    print("このセルは Google Colab 上でのみ動作します。")

## 4. 応用①：インスタンスセグメンテーション

`-seg` 付きのモデルは、枠に加えて物体ごとの**画素単位のマスク**を出力します（講義スライドの「インスタンスセグメンテーション」）。結果は `r.masks` に入っています。

In [ ]:
seg_model = YOLO("yolo26n-seg.pt")
rs = seg_model(BUS, device=DEVICE, verbose=False)[0]
show(rs.plot(), "YOLO26n-seg")

print("マスクの形 (物体数, 高さ, 幅):", tuple(rs.masks.data.shape))
print("1つ目の物体の輪郭の頂点数:", len(rs.masks.xy[0]))

### マスクの面積と、枠との違い

枠の面積とマスクの面積を比べると、枠がどれだけ「背景」を含んでいるかがわかります。マスクはモデルの入力解像度で出力されるため、元画像のサイズに合わせて拡大してから数えます。

In [ ]:
H, W = rs.orig_shape
masks = rs.masks.data.cpu().numpy()
rows = []
for i in range(len(masks)):
    m_full = cv2.resize(masks[i], (W, H), interpolation=cv2.INTER_NEAREST) > 0.5
    x1, y1, x2, y2 = rs.boxes.xyxy[i].cpu().numpy()
    box_area = (x2 - x1) * (y2 - y1)
    rows.append({"class": rs.names[int(rs.boxes.cls[i])],
                 "mask px": int(m_full.sum()),
                 "box px": int(box_area),
                 "mask / box": round(m_full.sum() / box_area, 2)})
pd.DataFrame(rows)

In [ ]:
# 人物のマスクだけを使って、背景をぼかす（簡易ポートレートモード）
img = cv2.imread(BUS)
person_mask = np.zeros((H, W), dtype=bool)
for i in range(len(masks)):
    if rs.names[int(rs.boxes.cls[i])] == "person":
        person_mask |= cv2.resize(masks[i], (W, H), interpolation=cv2.INTER_NEAREST) > 0.5

blurred = cv2.GaussianBlur(img, (51, 51), 0)
portrait = np.where(person_mask[..., None], img, blurred)
show(portrait, "Background blurred using person masks")

## 5. 応用②：姿勢推定

`-pose` 付きのモデルは、人物ごとに **17個の関節点（キーポイント）** を出力します。並び順は COCO 形式です。

In [ ]:
pose_model = YOLO("yolo26n-pose.pt")
rpo = pose_model(ZIDANE, device=DEVICE, verbose=False)[0]
show(rpo.plot(), "YOLO26n-pose", size=10)

KPT_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
             "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
             "left_wrist", "right_wrist", "left_hip", "right_hip",
             "left_knee", "right_knee", "left_ankle", "right_ankle"]
print("キーポイントの形 (人数, 17, xy):", tuple(rpo.keypoints.xy.shape))

In [ ]:
# 1人目のキーポイントを表にする（conf が低い点は画面外・隠れている可能性が高い）
kp = rpo.keypoints.xy[0].cpu().numpy()
kc = rpo.keypoints.conf[0].cpu().numpy()
pd.DataFrame({"keypoint": KPT_NAMES, "x": kp[:, 0].astype(float).round(1),
              "y": kp[:, 1].astype(float).round(1), "conf": kc.astype(float).round(2)})

### 関節の角度を計算する

肩・肘・手首の3点から肘の曲がり角度を求めます。フォーム解析やリハビリ支援などの応用の基本になる計算です。

In [ ]:
def angle(a, b, c):
    # 点 b を頂点とする角 abc（度）
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

rows = []
for p in range(len(rpo.keypoints.xy)):
    xy = rpo.keypoints.xy[p].cpu().numpy()
    cf = rpo.keypoints.conf[p].cpu().numpy()
    for side, (s, e, w) in {"left": (5, 7, 9), "right": (6, 8, 10)}.items():
        ok = min(cf[s], cf[e], cf[w]) > 0.5
        rows.append({"person": p, "side": side,
                     "elbow angle (deg)": round(float(angle(xy[s], xy[e], xy[w])), 1) if ok else None,
                     "reliable": ok})
pd.DataFrame(rows)

## 6. 応用③：動画の物体追跡

検出は1フレームずつ独立に行われますが、**追跡（tracking）** はフレーム間で同じ物体に同じ ID を付け続けます。ここでは追跡アルゴリズム **ByteTrack** を使います。

手元に動画がなくても試せるよう、まずサンプル画像の上をカメラが横に動くような**疑似動画**を作ります。自分の動画を使う場合は、6-4 を参照してください。

In [ ]:
# 疑似動画の作成：bus.jpg を左から右へパンする 3 秒（15fps × 45 フレーム）の動画
img = cv2.imread(BUS)
Hh, Ww = img.shape[:2]
VIDEO = "pan.mp4"
writer = cv2.VideoWriter(VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), 15, (640, 480))
N_FRAMES = 45
for t in range(N_FRAMES):
    x0 = int((Ww - 540) * t / (N_FRAMES - 1))
    frame = cv2.resize(img[300:900, x0:x0 + 540], (640, 480))
    writer.write(frame)
writer.release()
print("作成しました:", VIDEO)

### 6-1. 追跡を実行して ID を記録する

`model.track(..., persist=True)` で追跡します。`stream=True` にするとフレームを1枚ずつ受け取れるので、長い動画でもメモリを使いすぎません。

In [ ]:
track_model = YOLO("yolo26n.pt")
records, annotated = [], []
for f, res in enumerate(track_model.track(VIDEO, stream=True, persist=True, tracker="bytetrack.yaml",
                                          device=DEVICE, verbose=False)):
    annotated.append(res.plot())
    if res.boxes.id is None:
        continue
    for tid, cls, (cx, cy, w, h) in zip(res.boxes.id.int().tolist(), res.boxes.cls.int().tolist(),
                                       res.boxes.xywh.cpu().numpy()):
        records.append({"frame": f, "id": tid, "class": res.names[cls], "cx": cx, "cy": cy})

tracks = pd.DataFrame(records)
print("フレーム数:", len(annotated), " 追跡された ID:", sorted(tracks["id"].unique().tolist()))
tracks.groupby(["id", "class"]).frame.agg(["min", "max", "count"]).rename(
    columns={"min": "first frame", "max": "last frame", "count": "frames"})

### 6-2. いくつかのフレームを並べて確認する

同じ人物に同じ `id` が付き続けているかを確認しましょう。

In [ ]:
idx = np.linspace(0, len(annotated) - 1, 4).astype(int)
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, i in zip(axes, idx):
    ax.imshow(cv2.cvtColor(annotated[i], cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6-3. 軌跡を描く

各 ID の中心座標をつなぐと、画面上での動き（軌跡）が見えます。カメラが右へ動いているので、物体は画面上では左へ流れていきます。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(cv2.cvtColor(annotated[-1], cv2.COLOR_BGR2RGB), alpha=0.4)
for tid, g in tracks.groupby("id"):
    ax.plot(g.cx, g.cy, "-o", markersize=3, label=f"id {tid} ({g['class'].iloc[0]})")
ax.legend(loc="lower left", fontsize=9)
ax.set_title("Trajectories of tracked objects")
ax.axis("off")
plt.show()

In [ ]:
# 追跡結果を動画として保存し、ノートブック上で再生する（ブラウザで再生できる H.264 形式に変換）
writer = cv2.VideoWriter("tracked_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 15, annotated[0].shape[1::-1])
for fr in annotated:
    writer.write(fr)
writer.release()
!ffmpeg -y -loglevel error -i tracked_raw.mp4 -vcodec libx264 -pix_fmt yuv420p tracked.mp4

from IPython.display import Video
Video("tracked.mp4", embed=True, width=640)

### 6-4. 自分の動画で試す（任意）

スマートフォンで撮った短い動画（10秒程度）をアップロードし、上の 6-1 以降のセルの `VIDEO` をそのファイル名に変えて実行してください。長い動画は時間がかかるので、GPU ランタイムを使いましょう。

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    if up:
        VIDEO = list(up)[0]
        print("VIDEO =", VIDEO, "に設定しました。6-1 のセルから再実行してください。")
except ImportError:
    print("このセルは Google Colab 上でのみ動作します。")

## 7. 応用④：内蔵カメラでリアルタイム物体認識

PC の Web カメラやスマートフォンのカメラの映像を、その場で YOLO に通します。

**仕組み**：Colab の Python はクラウド上のサーバーで動いているため、手元のカメラに直接アクセスできません。そこで、ブラウザ側の JavaScript でカメラ映像を1フレームずつ JPEG にしてサーバーへ送ります。サーバー側の Python が YOLO で検出し、結果の画像をブラウザへ返します。これを繰り返すことで、疑似的にリアルタイム表示します。

```
[ブラウザ] カメラ → JPEG ──送信──▶ [Colab サーバー] YOLO で検出
     ▲                                          │
     └──────────── 結果画像を返す ◀──────────────┘
```

**注意**
- 最初にブラウザがカメラの使用許可を求めるので「許可」を選んでください。
- 通信を挟むため、フレームレートは GPU ランタイムで数 fps〜10 fps 程度です（回線速度に依存）。
- **スマートフォン**：スマートフォンのブラウザ（Chrome / Safari）で Colab を開いて実行できます。背面カメラを使うには、下の `FACING` を `"environment"` にしてください。
- 映像はブラウザとこの Colab セッションの間でのみやり取りされ、保存はされません（7-1 で撮影した画像を除く）。

In [ ]:
# ===== カメラ設定 =====
FACING = "user"        # "user" = 前面（PCの内蔵カメラ・自撮り側） / "environment" = スマホの背面カメラ
CAPTURE_WIDTH = 640    # 送信する画像の幅（小さいほど速い）
JPEG_QUALITY = 0.7     # 送信する JPEG の画質（0〜1）

import base64, json, time
from collections import Counter
from IPython.display import display, Javascript, Image as IPyImage

try:
    from google.colab.output import eval_js
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("このパートは Google Colab 上でのみ動作します（ブラウザのカメラを使うため）。")

def dataurl_to_bgr(data_url):
    # "data:image/jpeg;base64,...." → OpenCV 画像（BGR）
    jpg = base64.b64decode(data_url.split(",", 1)[1])
    return cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)

def bgr_to_dataurl(img_bgr, quality=80):
    ok, buf = cv2.imencode(".jpg", img_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()

### 7-1. まずは1枚撮影して検出する

「撮影」ボタンを押すと、その瞬間の画像で物体検出を行います。カメラが正しく動くかの確認にも使えます。

In [ ]:
TAKE_PHOTO_JS = '''
async function takePhoto(facing, width, quality) {
  const div = document.createElement('div');
  const video = document.createElement('video');
  video.setAttribute('playsinline', '');   // iPhone の Safari で全画面再生にならないようにする
  video.muted = true;
  video.style.width = '100%';
  video.style.maxWidth = width + 'px';
  video.style.display = 'block';
  const btn = document.createElement('button');
  btn.textContent = '撮影';
  btn.style.fontSize = '18px';
  btn.style.margin = '8px 0';
  div.appendChild(video);
  div.appendChild(btn);
  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: facing}, audio: false});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

  await new Promise((resolve) => btn.onclick = resolve);

  const scale = width / video.videoWidth;
  const canvas = document.createElement('canvas');
  canvas.width = width;
  canvas.height = Math.round(video.videoHeight * scale);
  canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
  stream.getVideoTracks().forEach(t => t.stop());
  div.remove();
  return canvas.toDataURL('image/jpeg', quality);
}
'''

if IN_COLAB:
    display(Javascript(TAKE_PHOTO_JS))
    data_url = eval_js(f"takePhoto('{FACING}', {CAPTURE_WIDTH}, {JPEG_QUALITY})")
    photo = dataurl_to_bgr(data_url)
    cv2.imwrite("camera_photo.jpg", photo)

    rc = model(photo, device=DEVICE, verbose=False)[0]
    show(rc.plot(), "Camera snapshot")
    display(boxes_to_df(rc))

## 課題： リアルタイム物体認識

以下の仕様を元にリアルタイム物体認識をするプログラムを作成してください。

====================================

下のセルを実行すると、大きな画面に検出結果、右下の小さな画面にカメラの生映像が表示されます。上部には処理速度（FPS）と、検出した物体の数がクラスごとに表示されます。

**止め方**：「停止」ボタンを押すか、結果の画面をクリックします（`MAX_SECONDS` 秒たつと自動で止まります）。

`RT_MODEL` を `"yolo26n-seg.pt"`（セグメンテーション）や `"yolo26n-pose.pt"`（姿勢推定）に変えると、ほかのタスクもリアルタイムで試せます。

In [ ]:
RT_MODEL = "yolo26n.pt"   # "yolo26n-seg.pt" や "yolo26n-pose.pt" に変えてもよい
RT_CONF = 0.4             # 確信度しきい値
MAX_SECONDS = 120         # 自動停止までの秒数

STREAM_JS = '''
var rt = null;   // カメラや表示要素をまとめて保持する

async function rtSetup(facing, width) {
  const div = document.createElement('div');
  div.style.position = 'relative';
  div.style.width = '100%';
  div.style.maxWidth = width + 'px';

  const label = document.createElement('div');
  label.style.cssText = 'font: 16px sans-serif; padding: 6px 8px; background: #13202E; color: #F3F1EA;';
  label.textContent = 'カメラを起動しています…';

  const result = document.createElement('img');    // サーバーから返ってきた検出結果
  result.style.cssText = 'width: 100%; display: block; background: #222; min-height: 240px; cursor: pointer;';
  result.title = 'クリックで停止';

  const video = document.createElement('video');   // 生のカメラ映像（右下に小さく表示）
  video.setAttribute('playsinline', '');
  video.muted = true;
  video.style.cssText = 'position: absolute; right: 8px; bottom: 56px; width: 25%; border: 2px solid #F2A25C;';

  const btn = document.createElement('button');
  btn.textContent = '停止';
  btn.style.cssText = 'font-size: 18px; margin: 8px 0;';

  div.appendChild(label);
  div.appendChild(result);
  div.appendChild(video);
  div.appendChild(btn);
  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: facing}, audio: false});
  video.srcObject = stream;
  await video.play();

  const canvas = document.createElement('canvas');
  canvas.width = width;
  canvas.height = Math.round(video.videoHeight * width / video.videoWidth);

  rt = {div, label, result, video, stream, canvas, stopped: false};
  btn.onclick = () => { rt.stopped = true; };
  result.onclick = () => { rt.stopped = true; };
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
}

// Python から毎回呼ばれる：前回の検出結果を表示し、次のフレームを返す
async function rtFrame(labelText, resultDataUrl, facing, width, quality) {
  if (rt === null) { await rtSetup(facing, width); }
  if (labelText) { rt.label.textContent = labelText; }
  if (resultDataUrl) { rt.result.src = resultDataUrl; }
  if (rt.stopped) {
    rtFinish();
    return '';
  }
  await new Promise(requestAnimationFrame);   // 画面の更新を1回待つ
  rt.canvas.getContext('2d').drawImage(rt.video, 0, 0, rt.canvas.width, rt.canvas.height);
  return rt.canvas.toDataURL('image/jpeg', quality);
}

// カメラを止めて後片付けをする（最後の検出結果の画像は残す）
function rtFinish() {
  if (rt === null) { return; }
  rt.stream.getVideoTracks().forEach(t => t.stop());
  rt.label.textContent = '停止しました';
  rt.video.remove();
  rt = null;
}
'''

def detect_frame(model, frame):
    # 1フレームを検出し、（描画済み画像, クラスごとの個数）を返す
    r = model(frame, conf=RT_CONF, device=DEVICE, verbose=False)[0]
    counts = Counter(r.names[int(c)] for c in r.boxes.cls.tolist())
    return r.plot(), counts

if IN_COLAB:
    rt_model = YOLO(RT_MODEL)
    display(Javascript(STREAM_JS))

    label, result_url = "", ""
    t_start = time.time()
    times = []
    n_frames = 0
    try:
        while time.time() - t_start < MAX_SECONDS:
            frame_url = eval_js(f"rtFrame({json.dumps(label)}, {json.dumps(result_url)}, "
                                f"'{FACING}', {CAPTURE_WIDTH}, {JPEG_QUALITY})")
            if not frame_url:          # 停止ボタンが押された
                break
            frame = dataurl_to_bgr(frame_url)
            annotated, counts = detect_frame(rt_model, frame)
            result_url = bgr_to_dataurl(annotated)
            n_frames += 1

            times.append(time.time())
            times = times[-10:]        # 直近10フレームで FPS を計算
            fps = (len(times) - 1) / (times[-1] - times[0]) if len(times) > 1 else 0.0
            objs = ", ".join(f"{k}×{v}" for k, v in counts.most_common()) or "なし"
            label = f"FPS {fps:.1f} | 検出: {objs} | 停止するには画面をクリック"
    finally:
        eval_js("rtFinish()")     # 時間切れや ■ で中断したときもカメラを確実に止める
        elapsed = time.time() - t_start
        print(f"処理したフレーム数: {n_frames}  平均 {n_frames / max(elapsed, 1e-6):.1f} fps")

### 参考：うまく動かないとき

| 症状 | 対処 |
|---|---|
| カメラの許可を求められない／黒い画面のまま | ブラウザのアドレスバー横の鍵アイコンからカメラを「許可」にし、セルを再実行する |
| `NotReadableError` などのエラー | Zoom や Teams など、ほかのアプリがカメラを使っていないか確認する |
| 動作が遅い | GPU ランタイムにする、`CAPTURE_WIDTH` を 480 や 320 に下げる |
| スマホで背面カメラにならない | `FACING = "environment"` にして設定セルから再実行する |
| セルが止まらない | 「停止」ボタンを押す。反応しない場合はツールバーの ■（実行を中断）を押す |